In [ ]:
from jointfm_client import bootstrap_notebook
bootstrap_notebook(add_src_root=True)

# Quantile Forecast
Forecast USD portfolio NAV and risk from 100 positive float daily observations: equity index level, 10-year Treasury yield, and one EUR/USD FX rate. Request quantile forecasts for portfolio NAV and realized volatility over the next 10 ordinal horizons.

In [ ]:
from pathlib import Path

import pandas as pd

from jointfm_client import JointFMClient, plan_forecast_columns

HISTORY_PATH = Path('notebooks/history.csv')
FEATURE_COLUMNS = ['equity_index_level', 'treasury_10y_yield', 'eur_usd_rate']
TARGET_COLUMNS = ['portfolio_nav', 'realized_volatility']
QUANTILES = [0.1, 0.3, 0.5, 0.7,0.9]
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(f'Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}')
if len(history) != INPUT_STEPS:
    raise ValueError(f'Expected {INPUT_STEPS} history rows, got {len(history)}')

client = JointFMClient.from_env()
plan = plan_forecast_columns(
    health=client.health(cache=True),
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)
result = client.forecast_quantiles(
    history,
    query_times=QUERY_TIMES,
    requested_columns=plan.requested_columns,
    columns=plan.columns,
    quantiles=QUANTILES,
    seed=7,
)
forecast = result.to_pandas_tidy()
expected_forecast_rows = OUTPUT_HORIZONS * len(plan.requested_columns) * len(QUANTILES)
if len(forecast) != expected_forecast_rows:
    raise ValueError(f'Expected {expected_forecast_rows} forecast rows, got {len(forecast)}')
forecast